In [1]:
# BiocManager::install("DropletUtils")

In [2]:
# Setup

library(Seurat)
library(DropletUtils)
library(Matrix)
library(jsonlite)
library(dplyr)
library(tidyr)

output_dir <- "./test_bench/Synthetic Visium"

dir.create(output_dir, showWarnings = FALSE)
spatial_dir <- file.path(output_dir, "spatial")
dir.create(spatial_dir, showWarnings = FALSE)

Loading required package: SeuratObject

Loading required package: sp


Attaching package: ‘SeuratObject’


The following objects are masked from ‘package:base’:

    intersect, t


Loading required package: SingleCellExperiment

Loading required package: SummarizedExperiment

Loading required package: MatrixGenerics

Loading required package: matrixStats


Attaching package: ‘MatrixGenerics’


The following objects are masked from ‘package:matrixStats’:

    colAlls, colAnyNAs, colAnys, colAvgsPerRowSet, colCollapse,
    colCounts, colCummaxs, colCummins, colCumprods, colCumsums,
    colDiffs, colIQRDiffs, colIQRs, colLogSumExps, colMadDiffs,
    colMads, colMaxs, colMeans2, colMedians, colMins, colOrderStats,
    colProds, colQuantiles, colRanges, colRanks, colSdDiffs, colSds,
    colSums2, colTabulates, colVarDiffs, colVars, colWeightedMads,
    colWeightedMeans, colWeightedMedians, colWeightedSds,
    colWeightedVars, rowAlls, rowAnyNAs, rowAnys, rowAvgsPerColSet,
    rowCollapse, r

In [3]:
Seurat <- readRDS("./Visium/Seurat_Clusters.rds")

In [4]:
# Config 

# set.seed(1) 

n_rows <- 50  
n_cols <- 50
spot_diameter <- 65 
fiducial_frame <- 100

mean_cells <- 5 
sd_cells <- 3    
min_cells <- 1   
max_cells <- 10  

In [5]:
# Generate Visium Grid

cat("Generating spatial coordinates...\n")

spots <- list()
spot_idx <- 1

for (row in 1:n_rows) {
  for (col in 1:n_cols) {
    row_px <- row * 100 
    col_px <- col * 100
    if (row %% 2 == 1) {
      col_px <- col_px + 50
    }
    
    barcode <- paste0("SYNTH-", row, "-", col, "-1")
    
    spots[[spot_idx]] <- data.frame(
      barcode = barcode,
      tissue = 1, # in_tissue
      row = row,
      col = col,
      imagerow = row_px,
      imagecol = col_px
    )
    spot_idx <- spot_idx + 1
  }
}
spatial_meta <- do.call(rbind, spots)
rownames(spatial_meta) <- spatial_meta$barcode

Generating spatial coordinates...


In [6]:
# Aggregate cells into spots
cat("Aggregating single cells into pseudo-spots...\n")

raw_counts <- GetAssayData(Seurat, layer = "counts") 
all_cell_ids <- colnames(Seurat)

n_spots_total <- nrow(spatial_meta)

# Simulate cells per spot using a truncated normal distribution
cells_per_spot_vec <- round(rnorm(n_spots_total, mean = mean_cells, sd = sd_cells))
cells_per_spot_vec <- pmax(min_cells, pmin(max_cells, cells_per_spot_vec)) 

spot_barcodes <- spatial_meta$barcode
cell_mapping_i <- c()
cell_mapping_j <- c() 
cell_mapping_x <- c()

truth_df_list <- list()

for (i in 1:n_spots_total) {
  n_to_sample <- cells_per_spot_vec[i]
  
  # Sample single cells with replacement
  chosen_cells <- sample(all_cell_ids, n_to_sample, replace = TRUE) 
    
  cell_indices <- match(chosen_cells, all_cell_ids)
  cell_mapping_i <- c(cell_mapping_i, cell_indices)
  cell_mapping_j <- c(cell_mapping_j, rep(i, length(cell_indices)))
  cell_mapping_x <- c(cell_mapping_x, rep(1, length(cell_indices)))
  
  cell_types <- Seurat@meta.data[chosen_cells, "CellType"]
  truth_df_list[[i]] <- data.frame(
    Spot = spot_barcodes[i],
    CellID = chosen_cells,
    CellType = cell_types
  )
}

# Create sparse mapping matrix for efficient aggregation
mapping_mat <- sparseMatrix(
  i = cell_mapping_i,
  j = cell_mapping_j,
  x = cell_mapping_x,
  dims = c(length(all_cell_ids), n_spots_total)
)

# Matrix multiplication to create composite spot profiles
visium_counts <- raw_counts %*% mapping_mat
colnames(visium_counts) <- spot_barcodes

cat("Synthetic Visium Matrix created with dimensions:", dim(visium_counts), "\n")

cat("Exporting ground truth cell counts...\n")
truth_df <- do.call(rbind, truth_df_list)

# 1. Calculate overall Total Cells per spot
spot_totals <- truth_df %>% 
  group_by(Spot) %>% 
  summarise(Total_Cells = n(), .groups = 'drop')

# 2. Count distinct cell types per spot and pivot to broad format
spot_types <- truth_df %>%
  group_by(Spot, CellType) %>%
  summarise(count = n(), .groups = 'drop') %>%
  pivot_wider(names_from = CellType, values_from = count, values_fill = 0)

# 3. Merge totals with the broad cell type table
spot_counts <- left_join(spot_totals, spot_types, by = "Spot")

# Write out the new expanded table
write.csv(spot_counts, file.path(output_dir, "ground_truth_counts.csv"), row.names = FALSE, quote = FALSE)
cat("Ground truth saved!\n")
cat("Done!\n")

Aggregating single cells into pseudo-spots...
Synthetic Visium Matrix created with dimensions: 36601 2500 
Exporting ground truth cell counts...
Ground truth saved!
Done!


In [7]:
# Write In Silico Visium data

cat("Writing authentic 10x files...\n")

write10xCounts(
  path = file.path(output_dir, "filtered_feature_bc_matrix.h5"),
  x = visium_counts,
  type = "HDF5",
  genome = "GRCh38", 
  version = "3"
)

tissue_pos_export <- data.frame(
  barcode = spatial_meta$barcode,
  in_tissue = spatial_meta$tissue,
  array_row = spatial_meta$row,
  array_col = spatial_meta$col,
  pxl_row_in_fullres = spatial_meta$imagerow,
  pxl_col_in_fullres = spatial_meta$imagecol
)

write.csv(tissue_pos_export, file.path(spatial_dir, "tissue_positions.csv"), row.names = FALSE, quote = FALSE)

scalefactors <- list(
  spot_diameter_fullres = 65,
  tissue_hires_scalef = 1.0,
  fiducial_diameter_fullres = 100,
  tissue_lowres_scalef = 1.0
)
write_json(scalefactors, file.path(spatial_dir, "scalefactors_json.json"), auto_unbox = TRUE)

tryCatch({
  library(png)
  dummy_img <- matrix(1, nrow = 2000, ncol = 2000)
  writePNG(dummy_img, target = file.path(spatial_dir, "tissue_lowres_image.png"))
  writePNG(dummy_img, target = file.path(spatial_dir, "tissue_hires_image.png"))
}, error = function(e) {
  cat("Warning: 'png' package not found. Images not created (might cause warnings in Seurat).\n")
})

cat("Done! Output saved to:", output_dir, "\n")

Writing authentic 10x files...


You created a large dataset with compression and chunking.
The chunk size is equal to the dataset dimensions.
If you want to read subsets of the dataset, you should testsmaller chunk sizes to improve read times.

You created a large dataset with compression and chunking.
The chunk size is equal to the dataset dimensions.
If you want to read subsets of the dataset, you should testsmaller chunk sizes to improve read times.



Done! Output saved to: ./test_bench/Synthetic Visium 
